# Week 6 — Integrative Capstone Project
**Project:** Customer segmentation (K-Means) and spending prediction (Random Forest) on online-retail-style transaction data.

**⚠️ Important data note:** This notebook is built around the structure of the well-known UCI/Kaggle *Online Retail* dataset. That specific file wasn't reachable from the environment this was built in, so Section 1 below generates a synthetic transaction dataset with the identical column structure and the same realistic data-quality issues (missing customer IDs, cancelled orders, zero-price rows). All numbers from Section 2 onward are real, computed results — just on this synthetic stand-in rather than the original file.

**To use the real dataset:** download `Online Retail.xlsx` from the UCI Machine Learning Repository or Kaggle, replace Section 1 with `pd.read_excel("Online Retail.xlsx")`, and everything from Section 2 onward runs unchanged. Full write-up in `Week_6_Report.docx`.

## 1. Generate Synthetic Transaction Data
(Skip this cell and load your own file instead if you have the real Online Retail dataset.)

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n_customers = 950
n_products = 120
countries = ["United Kingdom"] * 70 + ["Germany"] * 8 + ["France"] * 7 + ["EIRE"] * 5 + \
            ["Spain"] * 4 + ["Netherlands"] * 3 + ["Belgium"] * 3

customer_ids = np.arange(10000, 10000 + n_customers)
customer_country = rng.choice(countries, size=n_customers)

product_codes = [f"{10000+i}" for i in range(n_products)]
product_desc = [f"PRODUCT ITEM {i:03d}" for i in range(n_products)]
product_price = np.round(rng.gamma(shape=2.0, scale=2.5, size=n_products) + 0.5, 2)

propensity = rng.pareto(a=1.5, size=n_customers) + 0.1

rows = []
invoice_counter = 536000
start_date = pd.Timestamp("2010-12-01")
end_date = pd.Timestamp("2011-12-09")
date_range_days = (end_date - start_date).days

for idx, cust in enumerate(customer_ids):
    n_invoices = max(1, int(rng.poisson(lam=1 + propensity[idx] * 1.5)))
    for _ in range(n_invoices):
        invoice_counter += 1
        invoice_no = str(invoice_counter)
        is_cancel = rng.random() < 0.02
        if is_cancel:
            invoice_no = "C" + invoice_no
        n_lines = rng.integers(1, 8)
        invoice_day = start_date + pd.Timedelta(days=int(rng.integers(0, date_range_days)))
        invoice_time = pd.Timedelta(hours=int(rng.integers(8, 18)), minutes=int(rng.integers(0, 60)))
        invoice_dt = invoice_day + invoice_time
        for _ in range(n_lines):
            p_idx = rng.integers(0, n_products)
            qty = int(rng.integers(1, 20))
            if is_cancel:
                qty = -qty
            unit_price = product_price[p_idx]
            cust_val = cust if rng.random() > 0.03 else np.nan
            rows.append((
                invoice_no, product_codes[p_idx], product_desc[p_idx],
                qty, invoice_dt, unit_price, cust_val, customer_country[idx]
            ))

df = pd.DataFrame(rows, columns=[
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country"
])

bad_idx = rng.choice(df.index, size=40, replace=False)
df.loc[bad_idx, "UnitPrice"] = 0.0

df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
print(df.shape)
df.head()

(17553, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536508,10065,PRODUCT ITEM 065,7,2011-08-08 14:17:00,1.91,10079.0,United Kingdom
1,538510,10003,PRODUCT ITEM 003,9,2011-12-02 10:39:00,1.35,10560.0,United Kingdom
2,538395,10022,PRODUCT ITEM 022,17,2011-01-24 17:11:00,0.98,10523.0,France
3,540217,10052,PRODUCT ITEM 052,18,2011-07-18 12:02:00,3.22,10944.0,United Kingdom
4,538375,10088,PRODUCT ITEM 088,13,2010-12-09 14:47:00,12.33,10523.0,France


## 2. Load and Inspect

In [2]:
print(df.shape)
print(df.isnull().sum())

(17553, 8)
InvoiceNo        0
StockCode        0
Description      0
Quantity         0
InvoiceDate      0
UnitPrice        0
CustomerID     541
Country          0
dtype: int64


## 3. Data Cleaning

In [3]:
before_rows = len(df)
df = df.drop_duplicates()
df = df.dropna(subset=["CustomerID", "Quantity", "UnitPrice"])
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]
after_rows = len(df)

print(f"Rows before cleaning: {before_rows}, after cleaning: {after_rows}")

df["Sales"] = df["Quantity"] * df["UnitPrice"]

Rows before cleaning: 17553, after cleaning: 16621


## 4. Exploratory Analysis

In [4]:
print(df["Sales"].describe())

country_sales = df.groupby("Country")["Sales"].sum().sort_values(ascending=False)
print(country_sales)

count    16621.000000
mean        54.583314
std         50.226769
min          0.730000
25%         18.240000
50%         40.200000
75%         74.800000
max        349.220000
Name: Sales, dtype: float64
Country
United Kingdom    542878.15
Germany           150869.97
France             95296.66
Netherlands        48769.69
Spain              25927.46
EIRE               23354.57
Belgium            20132.77
Name: Sales, dtype: float64


## 5. Customer-Level Feature Engineering (RFM)

In [5]:
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
customer = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Sales", "sum")
).reset_index()

print(customer.shape)
customer.describe()

(943, 4)


,CustomerID,Recency,Frequency,Monetary
count,943.000000,943.000000,943.000000,943.000000
mean,10473.996819,121.387063,4.597031,962.067094
std,274.782325,102.200812,14.816961,3120.882198
min,10000.000000,1.000000,1.000000,1.250000
25%,10235.500000,32.000000,1.000000,187.065000
50%,10475.000000,93.000000,2.000000,432.790000
75%,10711.500000,190.500000,4.000000,901.750000
max,10949.000000,372.000000,244.000000,49984.820000


## 6. K-Means Customer Segmentation

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

features = customer[["Recency", "Frequency", "Monetary"]]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    print(k, round(km.inertia_, 1), round(silhouette_score(X_scaled, labels), 4))

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
customer["Cluster"] = kmeans.fit_predict(X_scaled)

cluster_profile = customer.groupby("Cluster")[["Recency", "Frequency", "Monetary"]].mean().round(2)
print(customer["Cluster"].value_counts().sort_index())
print(cluster_profile)

2 1283.3 0.925
3 557.3 0.579
4 381.9 0.5945
5 247.9 0.5121
6 181.0 0.5231
7 142.0 0.4838
Cluster
0    619
1      4
2    310
3     10
Name: count, dtype: int64
         Recency  Frequency  Monetary
Cluster                              
0          60.35       4.10    855.13
1           1.50     209.75  43980.62
2         248.30       1.55    318.89
3          13.40      47.70  10312.73


## 7. Random Forest — Spending Prediction

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = customer[["Recency", "Frequency"]]
y = customer["Monetary"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

regressor = RandomForestRegressor(n_estimators=200, random_state=42, max_depth=12, min_samples_leaf=2)
regressor.fit(X_train, y_train)
predictions = regressor.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions) ** 0.5
r2 = r2_score(y_test, predictions)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2:", round(r2, 4))

MAE: 294.8
RMSE: 1058.59
R2: 0.9276


## 8. Cross-Validation and Feature Importance

In [8]:
cv_scores = cross_val_score(regressor, X, y, cv=5, scoring="r2")
print("CV R2 scores:", np.round(cv_scores, 4))
print("Mean CV R2:", round(cv_scores.mean(), 4))

print("Feature importances:", dict(zip(X.columns, np.round(regressor.feature_importances_, 4))))

CV R2 scores: [0.9332 0.8522 0.9841 0.9459 0.9329]
Mean CV R2: 0.9296
Feature importances: {'Recency': np.float64(0.0085), 'Frequency': np.float64(0.9915)}


## Conclusion
Four customer segments emerge, two of them (under 1.5% of customers) accounting for a disproportionate share of monetary value. The regression explains ~93% of spend variance (R² ≈ 0.93) using just Recency and Frequency, with Frequency doing almost all the work. See `Week_6_Report.docx` for the full interpretation, including why the MAE/RMSE gap matters.